<a href="https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u1_funciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u1_funciones.ipynb)

# Dif. U1 · Funciones
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 1 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. Qué es una función: tablas de valores $(x,y)$
2. Función como regla de asignación y como gráfica
3. Dominio y rango
4. Álgebra de apoyo sobre la marcha
5. Funciones lineal y cuadrática
6. Funciones exponencial, logarítmica y trigonométricas

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todo). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, calcula a mano el caso que se indica y escribe tu resultado. La calculadora hace lo que le dices, no lo que querías decirle; por eso nunca la uses sin haber comprobado un caso que ya sabes resolver.

**Etiqueta del repositorio.** `DIF-U1` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u1_funciones/NN_*.py`; los fragmentos que el libro imprime, de `dif/u1_funciones/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias y `sqrt(...)` para raíces.

## 0. Versiones e imports

In [1]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)
from sympy.calculus.util import continuous_domain, function_range

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.5 MB/s eta 0:00:00
Python 3.13.16 | numpy 2.1.3 | sympy 1.14.0 | matplotlib 3.10.0 | ipywidgets 7.7.1


In [2]:
# Símbolos y utilidades compartidas por todo el notebook
x, y, h, a = sp.symbols("x y h a", real=True)

_LOCAL = {"x": x, "h": h, "a": a, "sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp,
          "log": sp.log, "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor)


def parsear(texto, variables=("x",)):
    """Convierte un texto en expresión de sympy o explica qué salió mal."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: x**2 - 3*x + 1")
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(_LOCAL))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x como variable y, si hace falta, "
                         "sqrt, abs, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=_LOCAL, global_dict=dict(_GLOBAL),
                          transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa x como variable, * para multiplicar, "
                         "** o ^ para potencias y sqrt(...) para raíces.") from err
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, abs, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - set(variables)
    if libres:
        raise ValueError(f"Símbolos no permitidos aquí: {sorted(libres)}. Solo puedes usar {list(variables)}.")
    return expr


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    v = float(valor)
    if not math.isfinite(v):
        return str(v)
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def conjunto_a_texto(s):
    """Intervalos de sympy en notación de intervalos."""
    if s == sp.S.Reals:
        return "(-∞, ∞) = ℝ"
    if s == sp.S.EmptySet:
        return "∅ (vacío)"
    if isinstance(s, sp.Union):
        return " ∪ ".join(conjunto_a_texto(t) for t in sorted(s.args, key=lambda t: float(t.inf)))
    if isinstance(s, sp.Interval):
        f = lambda p: ("∞" if p == sp.oo else "-∞" if p == -sp.oo
                       else str(p) if p.is_Rational else "≈" + cifras(float(p), 4))
        return ("(" if s.left_open else "[") + f(s.inf) + ", " + f(s.sup) + (")" if s.right_open else "]")
    return str(s)


def comparar(mi_resultado, esperado):
    """Celda Contrasta: compara tu resultado a mano con el de la calculadora."""
    if mi_resultado is None or str(mi_resultado).strip() == "":
        print("Falta tu cálculo a mano. Escríbelo arriba y vuelve a ejecutar la celda.")
        return None
    try:
        mio = parsear(str(mi_resultado), variables=("x", "h", "a"))
        ok = sp.simplify(mio - sp.sympify(esperado)) == 0
    except ValueError as err:
        print(err); return None
    print("Coinciden." if ok else "No coinciden: revisa en qué paso te desviaste "
          "(¿sustituiste en todas las apariciones de x? ¿pusiste paréntesis?).")
    return ok

## 1. Qué es una función: tablas de valores $(x,y)$

Una tabla define a $y$ como función de $x$ cuando **cada** valor de $x$ aparece con un solo valor de $y$. Dos $x$ distintos sí pueden compartir el mismo $y$.

La calculadora siguiente construye la tabla de una regla $f$ en $n$ valores igualmente espaciados y calcula las diferencias primera y segunda. Si la primera diferencia es constante, los puntos están sobre una recta; si lo es la segunda, sobre una parábola (siempre que los datos sean exactos).

In [3]:
def tabla_valores(texto, x0, x1, n, n_cifras=4):
    """Devuelve (xs, ys) con ys=None donde f no está definida. Valida las entradas."""
    f = parsear(texto)
    if not (math.isfinite(x0) and math.isfinite(x1)) or x0 >= x1:
        raise ValueError("El valor inicial de x debe ser menor que el final.")
    if not 2 <= int(n) <= 50:
        raise ValueError("Elige entre 2 y 50 puntos.")
    fn = sp.lambdify(x, f, "numpy")
    xs = np.linspace(x0, x1, int(n))
    with np.errstate(all="ignore"):
        ys = fn(xs)
    ys = np.full_like(xs, float(ys)) if np.ndim(ys) == 0 else np.asarray(ys, dtype=float)
    return xs, np.where(np.isfinite(ys), ys, np.nan)


def calculadora_tabla(texto, x0, x1, n, n_cifras):
    try:
        xs, ys = tabla_valores(texto, x0, x1, n)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {texto}      (valores redondeados a {n_cifras} cifras significativas)")
    print(f"{'x':>10} | {'y':>12}")
    for u, v in zip(xs, ys):
        print(f"{cifras(u, n_cifras):>10} | {('no definida' if np.isnan(v) else cifras(v, n_cifras)):>12}")
    ok = ~np.isnan(ys)
    if ok.all() and len(ys) >= 4:
        d1, d2 = np.diff(ys), np.diff(ys, 2)
        print("\nprimera diferencia constante :", bool(np.allclose(d1, d1[0])))
        print("segunda diferencia constante :", bool(np.allclose(d2, d2[0])))
    plt.figure(figsize=(4.5, 3)); plt.plot(xs[ok], ys[ok], "o"); plt.grid(True)
    plt.xlabel("x"); plt.ylabel("y"); plt.title("Puntos de la tabla"); plt.show()


widgets.interact(calculadora_tabla,
    texto=widgets.Text(value="x**2 + 1", description="f(x) ="),
    x0=widgets.FloatText(value=0.0, description="x inicial"),
    x1=widgets.FloatText(value=4.0, description="x final"),
    n=widgets.IntSlider(value=5, min=2, max=30, description="puntos"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=10, description="cifras sig."));

interactive(children=(Text(value='x**2 + 1', description='f(x) ='), FloatText(value=0.0, description='x inicia…

In [4]:
def texto_a_pares(texto):
    """'0:20, 5:55, 5:60' -> [(0.0, 20.0), (5.0, 55.0), (5.0, 60.0)]"""
    pares = []
    for token in texto.replace(";", ",").split(","):
        if not token.strip():
            continue
        if token.count(":") != 1:
            raise ValueError(f"'{token.strip()}' no tiene el formato x:y (por ejemplo 5:55).")
        u, v = token.split(":")
        try:
            pares.append((float(u), float(v)))
        except ValueError:
            raise ValueError(f"'{token.strip()}' no es un par de números.") from None
    if not pares:
        raise ValueError("Escribe al menos un par x:y.")
    return pares


def es_funcion(pares):
    """(True, []) si cada x aparece con un solo y; si no, (False, pares en conflicto)."""
    vistos, conflictos = {}, []
    for u, v in pares:
        if u in vistos and vistos[u] != v:
            conflictos.append(((u, vistos[u]), (u, v)))
        vistos.setdefault(u, v)
    return len(conflictos) == 0, conflictos


def verificador_tabla(texto):
    try:
        pares = texto_a_pares(texto)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for etiqueta, par in (("y es función de x", pares), ("x es función de y", [(v, u) for u, v in pares])):
        ok, conf = es_funcion(par)
        print(f"{etiqueta}: {'SÍ' if ok else 'NO'}", "" if ok else f"  (conflicto: {conf[0]})")
    print("Nota: con datos medidos, antes de concluir compara la diferencia entre lecturas "
          "con la incertidumbre del instrumento.")


widgets.interact(verificador_tabla, texto=widgets.Text(
    value="0:20, 5:55, 5:60, 10:88", description="pares x:y", layout=widgets.Layout(width="480px")));

interactive(children=(Text(value='0:20, 5:55, 5:60, 10:88', description='pares x:y', layout=Layout(width='480p…

In [5]:
# Casos de prueba de la sección 1 (resultado conocido)
def valor_en(texto, v):
    return float(parsear(texto).subs(x, v))

PRUEBAS_1 = [
    ("2*x + 1 en x=3 da 7",            lambda: valor_en("2*x + 1", 3) == 7),
    ("x**2 - 4 en x=-2 da 0",          lambda: valor_en("x**2 - 4", -2) == 0),
    ("3 - x en x=5 da -2",             lambda: valor_en("3 - x", 5) == -2),
    ("tabla repetida (1:4,2:6,2:7) no es función", lambda: not es_funcion(texto_a_pares("1:4, 2:6, 2:7"))[0]),
    ("la misma tabla sí define x como función de y", lambda: es_funcion([(v, u) for u, v in texto_a_pares("1:4, 2:6, 2:7")])[0]),
    ("y=x**2+1 en 0..4: segunda diferencia constante 2", lambda: np.allclose(np.diff(tabla_valores("x**2+1", 0, 4, 5)[1], 2), 2)),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

OK   2*x + 1 en x=3 da 7
OK   x**2 - 4 en x=-2 da 0
OK   3 - x en x=5 da -2
OK   tabla repetida (1:4,2:6,2:7) no es función
OK   la misma tabla sí define x como función de y
OK   y=x**2+1 en 0..4: segunda diferencia constante 2


**Contrasta (sección 1).** Calcula a mano $f(-2)$, $f(0)$ y $f(3)$ para $f(x)=x^2-2x$ y compara con la tabla de la calculadora (usa $x$ de $-2$ a $3$ con 6 puntos). Anota: ¿coincidieron? Si no, ¿qué paso falló?

In [6]:
# Contrasta 1: escribe tus tres valores calculados a mano
mi_f_menos2, mi_f_0, mi_f_3 = None, None, None     # por ejemplo: 8, 0, 3

esperado = [valor_en("x**2 - 2*x", v) for v in (-2, 0, 3)]
print("La calculadora da:", esperado)
for nombre, mio, ok in zip(("f(-2)", "f(0)", "f(3)"), (mi_f_menos2, mi_f_0, mi_f_3), esperado):
    print(nombre, "->", "falta tu cálculo" if mio is None else ("coincide" if abs(mio - ok) < 1e-9 else "NO coincide"))

La calculadora da: [8.0, 0.0, 3.0]
f(-2) -> falta tu cálculo
f(0) -> falta tu cálculo
f(3) -> falta tu cálculo


## 2. Función como regla de asignación y como gráfica

**Evaluar en una expresión.** Para $f(x)=x^2-3x+1$, el valor $f(a+1)$ se obtiene sustituyendo *cada* aparición de $x$ por $(a+1)$, con paréntesis. La primera calculadora lo hace con `sympy`.

**Recta vertical.** La segunda calculadora cuenta en cuántos puntos corta la recta $x=c$ a una curva. Una sola recta con dos puntos basta para descartar que la curva sea gráfica de una función; una recta con un punto no basta para afirmar que lo es (hay que probar todas).

In [7]:
def evaluar_expresion(texto_f, texto_arg):
    f = parsear(texto_f)
    arg = parsear(texto_arg, variables=("x", "h", "a"))
    return sp.expand(f.subs(x, arg))


def calculadora_evaluar(texto_f, texto_arg):
    try:
        res = evaluar_expresion(texto_f, texto_arg)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {texto_f}")
    print(f"f({texto_arg}) = {res}")


widgets.interact(calculadora_evaluar,
    texto_f=widgets.Text(value="x**2 - 3*x + 1", description="f(x) ="),
    texto_arg=widgets.Text(value="a + 1", description="evaluar en"));

interactive(children=(Text(value='x**2 - 3*x + 1', description='f(x) ='), Text(value='a + 1', description='eva…

In [8]:
RELACIONES = {
    "x² + y² = 25 (circunferencia)": sp.Eq(x**2 + y**2, 25),
    "y² = x": sp.Eq(y**2, x),
    "y = x²": sp.Eq(y, x**2),
    "x² + 4y² = 16 (elipse)": sp.Eq(x**2 + 4*y**2, 16),
}


def recta_vertical(nombre, c):
    """Valores de y donde la recta x=c corta la relación (todos, con su signo)."""
    sols = sp.solve(RELACIONES[nombre].subs(x, sp.nsimplify(c)), y)
    return sorted({float(s) for s in sols if sp.im(s) == 0}, reverse=True)


def calculadora_recta_vertical(nombre, c, n_cifras):
    ys = recta_vertical(nombre, c)
    print(f"Relación: {nombre}      recta x = {cifras(c, n_cifras)}")
    print("Puntos de intersección:", [(cifras(c, n_cifras), cifras(v, n_cifras)) for v in ys] or "ninguno")
    if len(ys) > 1:
        print("Hay más de un punto: la curva NO es gráfica de y = f(x).")
    else:
        print("Con esta recta no hay violación. Eso no prueba que sea función: prueba otros valores de c.")
    t = np.linspace(-6, 6, 600)
    plt.figure(figsize=(4, 4)); plt.axvline(c, color="gray", ls="--")
    xs = np.linspace(-6, 6, 1200)
    for expr in sp.solve(RELACIONES[nombre], y):
        fn = sp.lambdify(x, expr, "numpy")
        with np.errstate(all="ignore"):
            yy = np.asarray(fn(xs), dtype=complex)
        yy = np.where(np.abs(yy.imag) < 1e-12, yy.real, np.nan)
        plt.plot(xs, yy, "b-")
    plt.plot([c]*len(ys), ys, "ro"); plt.xlim(-6, 6); plt.ylim(-6, 6); plt.grid(True)
    plt.gca().set_aspect("equal"); plt.xlabel("x"); plt.ylabel("y"); plt.show()


widgets.interact(calculadora_recta_vertical,
    nombre=widgets.Dropdown(options=list(RELACIONES), description="relación"),
    c=widgets.FloatSlider(value=3.0, min=-6, max=6, step=0.1, description="c"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

interactive(children=(Dropdown(description='relación', options=('x² + y² = 25 (circunferencia)', 'y² = x', 'y …

In [9]:
PRUEBAS_2 = [
    ("x²+y²=25 en x=3 da y=4 y y=-4",   lambda: recta_vertical("x² + y² = 25 (circunferencia)", 3) == [4.0, -4.0]),
    ("y²=x en x=0 da un solo punto",    lambda: recta_vertical("y² = x", 0) == [0.0]),
    ("y²=x en x=4 da y=2 y y=-2",       lambda: recta_vertical("y² = x", 4) == [2.0, -2.0]),
    ("y=x² en x=-1 da un solo punto",   lambda: recta_vertical("y = x²", -1) == [1.0]),
    ("f(a+1) para x²+2x-1 es a²+4a+2",  lambda: sp.simplify(evaluar_expresion("x**2 + 2*x - 1", "a + 1") - (a**2 + 4*a + 2)) == 0),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

OK   x²+y²=25 en x=3 da y=4 y y=-4
OK   y²=x en x=0 da un solo punto
OK   y²=x en x=4 da y=2 y y=-2
OK   y=x² en x=-1 da un solo punto
OK   f(a+1) para x²+2x-1 es a²+4a+2


**Contrasta (sección 2).** Calcula a mano $f(a+1)$ para $f(x)=x^2+2x-1$ y escríbelo en la celda. Anota: ¿coincide con la calculadora? Si no, ¿pusiste paréntesis al sustituir?

In [10]:
mi_resultado = ""      # por ejemplo: "a**2 + 4*a + 2"
comparar(mi_resultado, evaluar_expresion("x**2 + 2*x - 1", "a + 1"))

Falta tu cálculo a mano. Escríbelo arriba y vuelve a ejecutar la celda.


## 3. Dominio y rango

La calculadora devuelve el dominio y el rango con `sympy` y además barre $[-10,10]$ buscando puntos donde la regla no produce un número real. El barrido es una comprobación aproximada: confirma al dominio, no lo demuestra.

Condiciones que aplica la teoría: denominador distinto de cero y radicando de raíz de índice par mayor o igual que cero, sobre la expresión **sin simplificar**.

In [11]:
def dominio_rango(texto):
    f = parsear(texto)
    try:
        d = continuous_domain(f, x, sp.S.Reals)
    except NotImplementedError:
        d = None
    try:
        r = function_range(f, x, sp.S.Reals)
    except NotImplementedError:
        r = None
    return f, d, r


def calculadora_dominio(texto):
    try:
        f, d, r = dominio_rango(texto)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {texto}")
    print("Dominio:", "sympy no pudo determinarlo" if d is None else conjunto_a_texto(d))
    print("Rango  :", "sympy no pudo determinarlo" if r is None else conjunto_a_texto(r))
    fn = sp.lambdify(x, f, "numpy")
    xs = np.linspace(-10, 10, 2001)
    with np.errstate(all="ignore"):
        ys = fn(xs)
    ys = np.full_like(xs, float(ys)) if np.ndim(ys) == 0 else np.asarray(ys, dtype=float)
    malos = xs[~np.isfinite(ys)]
    print("Barrido en [-10, 10]:",
          "f produce valores reales en todos los puntos revisados." if malos.size == 0
          else f"f falla en {malos.size} de {xs.size} puntos revisados (de x ≈ {cifras(malos.min(), 4)} a x ≈ {cifras(malos.max(), 4)}).")
    plt.figure(figsize=(4.5, 3)); plt.plot(xs, np.where(np.isfinite(ys) & (np.abs(ys) < 20), ys, np.nan), "b-")
    plt.grid(True); plt.xlabel("x"); plt.ylabel("y"); plt.ylim(-6, 6); plt.show()


widgets.interact(calculadora_dominio, texto=widgets.Text(
    value="sqrt(x + 3)/(x**2 - 4)", description="f(x) =", layout=widgets.Layout(width="420px")));

interactive(children=(Text(value='sqrt(x + 3)/(x**2 - 4)', description='f(x) =', layout=Layout(width='420px'))…

In [12]:
def en_texto(f):
    _, d, r = dominio_rango(f)
    return conjunto_a_texto(d), conjunto_a_texto(r)

PRUEBAS_3 = [
    ("√(x+3): dominio [-3, ∞), rango [0, ∞)", lambda: en_texto("sqrt(x+3)") == ("[-3, ∞)", "[0, ∞)")),
    ("1/(x-2): dominio y rango sin 2 y sin 0",  lambda: en_texto("1/(x-2)") == ("(-∞, 2) ∪ (2, ∞)", "(-∞, 0) ∪ (0, ∞)")),
    ("√(9-x²): dominio [-3,3], rango [0,3]",    lambda: en_texto("sqrt(9-x**2)") == ("[-3, 3]", "[0, 3]")),
    ("√(x+3)/(x²-4): dominio [-3,-2) ∪ (-2,2) ∪ (2,∞)", lambda: en_texto("sqrt(x+3)/(x**2-4)")[0] == "[-3, -2) ∪ (-2, 2) ∪ (2, ∞)"),
    ("(x²-4)/(x-2) conserva el hueco en x=2",    lambda: en_texto("(x**2-4)/(x-2)")[0] == "(-∞, 2) ∪ (2, ∞)"),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

OK   √(x+3): dominio [-3, ∞), rango [0, ∞)
OK   1/(x-2): dominio y rango sin 2 y sin 0
OK   √(9-x²): dominio [-3,3], rango [0,3]
OK   √(x+3)/(x²-4): dominio [-3,-2) ∪ (-2,2) ∪ (2,∞)
OK   (x²-4)/(x-2) conserva el hueco en x=2


**Contrasta (sección 3).** Determina a mano el dominio de $f(x)=\dfrac{\sqrt{x-1}}{x-4}$. Escríbelo como intervalos, ejecuta la calculadora con esa expresión y compara. Anota: ¿incluiste el extremo $1$? ¿excluiste el $4$?

In [13]:
mi_dominio = ""        # por ejemplo: "[1, 4) ∪ (4, ∞)"
_, d, _ = dominio_rango("sqrt(x - 1)/(x - 4)")
print("La calculadora da:", conjunto_a_texto(d))
print("Tu resultado     :", mi_dominio or "(falta tu cálculo a mano)")

La calculadora da: [1, 4) ∪ (4, ∞)
Tu resultado     : (falta tu cálculo a mano)


## 4. Álgebra de apoyo sobre la marcha

La calculadora expande $f(x+h)$, simplifica el cociente de diferencias $\dfrac{f(x+h)-f(x)}{h}$ y comprueba que $f(x+h)$ **no** es $f(x)+h$. Después muestra por qué conviene simplificar antes de evaluar: con $h$ muy pequeño, restar dos números casi iguales destruye cifras significativas.

In [14]:
def cociente_diferencias(texto):
    f = parsear(texto)
    fxh = sp.expand(f.subs(x, x + h))
    cociente = sp.simplify((fxh - f) / h)
    es_mas_h = sp.simplify(fxh - (f + h)) == 0
    return f, fxh, cociente, es_mas_h


def calculadora_cociente(texto, x0, exponente_h):
    try:
        f, fxh, cociente, es_mas_h = cociente_diferencias(texto)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x)               = {f}")
    print(f"f(x+h) expandida   = {fxh}")
    print(f"[f(x+h)-f(x)]/h    = {cociente}      (válido para h ≠ 0)")
    print(f"¿f(x+h) = f(x)+h?  = {es_mas_h}")
    fn = sp.lambdify(x, f, "numpy"); simp = sp.lambdify((x, h), cociente, "numpy")
    print(f"\nEn x = {x0}: cociente directo contra cociente simplificado")
    print(f"{'h':>10} | {'directo':>14} | {'simplificado':>14}")
    for e in sorted({exponente_h, 1, 4, 8, 12, 16}):
        hh = 10.0**(-e)
        directo = (fn(x0 + hh) - fn(x0)) / hh
        print(f"{hh:>10.0e} | {directo:>14.8g} | {float(simp(x0, hh)):>14.8g}")


widgets.interact(calculadora_cociente,
    texto=widgets.Text(value="x**2 - 3*x", description="f(x) ="),
    x0=widgets.FloatText(value=4.0, description="x"),
    exponente_h=widgets.IntSlider(value=6, min=1, max=16, description="h = 10^-k"));

interactive(children=(Text(value='x**2 - 3*x', description='f(x) ='), FloatText(value=4.0, description='x'), I…

In [15]:
# Precaución de herramienta: la jerarquía de operaciones también se aplica en Python
for v in (4, 9):
    print(f"x = {v}:  x**1/2 = {v**1/2:<5}  x**(1/2) = {v**(1/2)}")
# En x = 4 los dos coinciden por casualidad (4/2 = 2 = raíz de 4); en x = 9 ya no.

x = 4:  x**1/2 = 2.0    x**(1/2) = 2.0
x = 9:  x**1/2 = 4.5    x**(1/2) = 3.0


In [16]:
PRUEBAS_4 = [
    ("x² -> 2x + h",            lambda: sp.simplify(cociente_diferencias("x**2")[2] - (2*x + h)) == 0),
    ("x³ -> 3x² + 3xh + h²",    lambda: sp.simplify(cociente_diferencias("x**3")[2] - (3*x**2 + 3*x*h + h**2)) == 0),
    ("2x + 1 -> 2",             lambda: sp.simplify(cociente_diferencias("2*x + 1")[2] - 2) == 0),
    ("x²-3x -> 2x + h - 3",     lambda: sp.simplify(cociente_diferencias("x**2 - 3*x")[2] - (2*x + h - 3)) == 0),
    ("f(x+h) no es f(x)+h para x²-3x", lambda: cociente_diferencias("x**2 - 3*x")[3] is False),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

OK   x² -> 2x + h
OK   x³ -> 3x² + 3xh + h²
OK   2x + 1 -> 2
OK   x²-3x -> 2x + h - 3
OK   f(x+h) no es f(x)+h para x²-3x


**Contrasta (sección 4).** Calcula a mano $\dfrac{f(x+h)-f(x)}{h}$ para $f(x)=x^2-3x$. Compara con la calculadora. Anota: ¿cancelaste $h$ como factor? ¿qué condición sobre $h$ necesitas para poder hacerlo?

In [17]:
mi_resultado = ""      # por ejemplo: "2*x + h - 3"
comparar(mi_resultado, cociente_diferencias("x**2 - 3*x")[2])

Falta tu cálculo a mano. Escríbelo arriba y vuelve a ejecutar la celda.


## 5. Funciones lineal y cuadrática

Dos calculadoras. La primera obtiene pendiente y ordenada al origen de la recta que pasa por dos puntos. La segunda analiza $f(x)=ax^2+bx+c$: discriminante, **ambas** raíces con su signo, vértice, concavidad y rango. Si $a=0$ avisa que la función ya no es cuadrática.

In [29]:
def recta_dos_puntos(x1, y1, x2, y2):
    if x1 == x2:
        raise ValueError("Con x1 = x2 la recta es vertical y no es gráfica de una función de x.")
    m = (y2 - y1) / (x2 - x1)
    return m, y1 - m * x1


def calculadora_recta(x1, y1, x2, y2, n_cifras):
    try:
        m, b = recta_dos_puntos(x1, y1, x2, y2)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"pendiente m = {cifras(m, n_cifras)}      ordenada al origen b = {cifras(b, n_cifras)}")
    print(f"y = {cifras(m, n_cifras)} x + {cifras(b, n_cifras)}")


widgets.interact(calculadora_recta,
    x1=widgets.FloatText(value=1.0, description="x1"), y1=widgets.FloatText(value=3.0, description="y1"),
    x2=widgets.FloatText(value=4.0, description="x2"), y2=widgets.FloatText(value=12.0, description="y2"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig."));

interactive(children=(FloatText(value=1.0, description='x1'), FloatText(value=3.0, description='y1'), FloatTex…

In [19]:
def cuadratica(A, B, C):
    """Análisis de a x² + b x + c. Devuelve un diccionario con todo lo que pide la teoría."""
    if A == 0:
        raise ValueError("a = 0: la función ya no es cuadrática, es la recta y = bx + c.")
    disc = B**2 - 4*A*C
    xv = -B / (2*A); yv = A*xv**2 + B*xv + C
    if disc > 0:
        raices = sorted([(-B + s*math.sqrt(disc)) / (2*A) for s in (1, -1)])
    elif disc == 0:
        raices = [xv]
    else:
        raices = []
    rango = f"[{cifras(yv, 6)}, ∞)" if A > 0 else f"(-∞, {cifras(yv, 6)}]"
    return {"disc": disc, "vertice": (xv, yv), "raices": raices, "rango": rango,
            "abre": "hacia arriba" if A > 0 else "hacia abajo", "f(0)": C}


def calculadora_cuadratica(A, B, C, n_cifras):
    try:
        r = cuadratica(A, B, C)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    xv, yv = r["vertice"]
    poli = sp.nsimplify(A)*x**2 + sp.nsimplify(B)*x + sp.nsimplify(C)
    print(f"f(x) = {poli}      abre {r['abre']}      f(0) = {cifras(r['f(0)'], n_cifras)}")
    print("discriminante:", cifras(r["disc"], n_cifras))
    if r["disc"] > 0:
        print("raíces (ambas):", ", ".join(cifras(v, n_cifras) for v in r["raices"]))
    elif r["disc"] == 0:
        print("una raíz real (doble):", cifras(r["raices"][0], n_cifras))
    else:
        print("sin raíces reales (el discriminante es negativo)")
    print("vértice:", (cifras(xv, n_cifras), cifras(yv, n_cifras)), "      rango:", r["rango"])
    ancho = max(2.0, abs(xv) + 3, *(abs(v - xv) + 1 for v in r["raices"]))
    xs = np.linspace(xv - ancho, xv + ancho, 400)
    plt.figure(figsize=(4.5, 3)); plt.plot(xs, A*xs**2 + B*xs + C, "b-")
    plt.plot([xv], [yv], "rs"); plt.plot(r["raices"], [0]*len(r["raices"]), "ko", mfc="white")
    plt.axvline(xv, color="gray", ls="--"); plt.axhline(0, color="k", lw=0.8); plt.grid(True)
    plt.xlabel("x"); plt.ylabel("y"); plt.show()


widgets.interact(calculadora_cuadratica,
    A=widgets.FloatText(value=1.0, description="a"), B=widgets.FloatText(value=-4.0, description="b"),
    C=widgets.FloatText(value=3.0, description="c"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

interactive(children=(FloatText(value=1.0, description='a'), FloatText(value=-4.0, description='b'), FloatText…

In [20]:
def cerca(u, v, tol=1e-9):
    return abs(u - v) < tol

PRUEBAS_5 = [
    ("x²-4x+3: raíces 1 y 3, vértice (2,-1)",
     lambda: [cerca(p, q) for p, q in zip(cuadratica(1, -4, 3)["raices"], (1, 3))] == [True, True]
             and cuadratica(1, -4, 3)["vertice"] == (2.0, -1.0)),
    ("-2x²+400x-8000: raíces 22.54 y 177.46, vértice (100, 12000)",
     lambda: [round(v, 2) for v in cuadratica(-2, 400, -8000)["raices"]] == [22.54, 177.46]
             and cuadratica(-2, 400, -8000)["vertice"] == (100.0, 12000.0)),
    ("x²+2x+5: discriminante -16, sin raíces reales",
     lambda: cuadratica(1, 2, 5)["disc"] == -16 and cuadratica(1, 2, 5)["raices"] == []),
    ("x²-6x+9: una raíz doble en 3",   lambda: cuadratica(1, -6, 9)["raices"] == [3.0]),
    ("recta por (1,3) y (4,12): m=3, b=0", lambda: recta_dos_puntos(1, 3, 4, 12) == (3.0, 0.0)),
    ("a=0 se rechaza con mensaje",     lambda: _rechaza(lambda: cuadratica(0, 1, 1))),
    ("x1=x2 se rechaza con mensaje",   lambda: _rechaza(lambda: recta_dos_puntos(2, 1, 2, 5))),
]

def _rechaza(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

OK   x²-4x+3: raíces 1 y 3, vértice (2,-1)
OK   -2x²+400x-8000: raíces 22.54 y 177.46, vértice (100, 12000)
OK   x²+2x+5: discriminante -16, sin raíces reales
OK   x²-6x+9: una raíz doble en 3
OK   recta por (1,3) y (4,12): m=3, b=0
OK   a=0 se rechaza con mensaje
OK   x1=x2 se rechaza con mensaje


**Contrasta (sección 5).** Calcula a mano el discriminante, las dos raíces y el vértice de $f(x)=x^2-6x+5$. Compara con la calculadora. Anota: ¿escribiste las dos raíces? ¿el vértice queda en el punto medio de ellas?

In [21]:
mis_raices = []        # por ejemplo: [1, 5]
mi_vertice = None      # por ejemplo: (3, -4)

r = cuadratica(1, -6, 5)
print("La calculadora da: raíces", r["raices"], "| vértice", r["vertice"])
print("Tus raíces  :", mis_raices or "(falta tu cálculo a mano)")
print("Tu vértice  :", mi_vertice or "(falta tu cálculo a mano)")
if mis_raices:
    print("Raíces:", "coinciden" if sorted(mis_raices) == r["raices"] else "NO coinciden")

La calculadora da: raíces [1.0, 5.0] | vértice (3.0, -4.0)
Tus raíces  : (falta tu cálculo a mano)
Tu vértice  : (falta tu cálculo a mano)


**Retoma tu predicción de la semilla.** Con los datos $(0\,^\circ\mathrm{C},\,0.50\ \mathrm{V})$ y $(30\,^\circ\mathrm{C},\,0.80\ \mathrm{V})$ de la tabla del sensor, obtén la recta $V(T)$ y calcula $V(25)$ y $V(100)$. Compara con lo que anotaste. La primera es una interpolación (dentro del intervalo medido); la segunda, una extrapolación, y solo vale si el sensor sigue siendo lineal fuera del rango calibrado.

In [22]:
m, b = recta_dos_puntos(0, 0.50, 30, 0.80)
print(f"V(T) = {m:.4g} T + {b:.4g}   [V, °C]")
print("V(25)  =", round(m*25 + b, 4), "V   (interpolación, dentro de 0 a 30 °C)")
print("V(100) =", round(m*100 + b, 4), "V  (extrapolación: supone que la recta sigue valiendo)")

V(T) = 0.01 T + 0.5   [V, °C]
V(25)  = 0.75 V   (interpolación, dentro de 0 a 30 °C)
V(100) = 1.5 V  (extrapolación: supone que la recta sigue valiendo)


## 6. Funciones exponencial, logarítmica y trigonométricas

Tres calculadoras. La primera da seno, coseno y tangente con la **unidad del ángulo visible** (grados o radianes). La segunda resuelve $a^x=c$ con logaritmos y explica cuándo no hay solución. La tercera describe una sinusoide $A\,\mathrm{sen}(\omega t+\varphi)+k$: amplitud, periodo, frecuencia y valor inicial.

**Convención.** En este libro $\ln$ es el logaritmo natural (base $e$) y $\log_{10}$ el decimal. En Python, `math.log(x)` y `np.log(x)` son el logaritmo **natural**; el decimal es `math.log10(x)`. Además, `math.sin`, `math.cos` y `np.sin` reciben **radianes**.

In [23]:
def trig_en(angulo, unidad):
    """sen, cos y tan de un ángulo. La unidad es obligatoria: 'grados' o 'radianes'."""
    if unidad not in ("grados", "radianes"):
        raise ValueError("La unidad debe ser 'grados' o 'radianes'.")
    t = math.radians(angulo) if unidad == "grados" else angulo
    s, c = math.sin(t), math.cos(t)
    s = 0.0 if abs(s) < 1e-12 else s          # lo menor que 1e-12 es ruido numérico, no un valor
    c = 0.0 if abs(c) < 1e-12 else c
    tg = s / c if c != 0 else None            # tan no existe donde cos = 0
    return s, c, tg


def calculadora_angulos(angulo, unidad, n_cifras):
    try:
        s, c, tg = trig_en(angulo, unidad)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    rad = math.radians(angulo) if unidad == "grados" else angulo
    print(f"ángulo = {cifras(angulo, n_cifras)} {unidad}  =  {cifras(rad, n_cifras)} rad  =  {cifras(math.degrees(rad), n_cifras)}°")
    print("sen =", cifras(s, n_cifras), "     cos =", cifras(c, n_cifras),
          "     tan =", cifras(tg, n_cifras) if tg is not None else "no existe (cos = 0)")
    print("sen² + cos² =", cifras(s**2 + c**2, n_cifras))


widgets.interact(calculadora_angulos,
    angulo=widgets.FloatText(value=30.0, description="ángulo"),
    unidad=widgets.Dropdown(options=["grados", "radianes"], value="grados", description="unidad"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

interactive(children=(FloatText(value=30.0, description='ángulo'), Dropdown(description='unidad', options=('gr…

In [24]:
def resolver_exponencial(base, c):
    """Resuelve base**x = c. Devuelve x o lanza ValueError con la razón."""
    if base <= 0 or base == 1:
        raise ValueError("La base debe cumplir base > 0 y base ≠ 1.")
    if c <= 0:
        raise ValueError("base**x siempre es positiva: con c ≤ 0 no hay solución real.")
    return math.log(c) / math.log(base)


def calculadora_exponencial(base, c, n_cifras):
    try:
        xs = resolver_exponencial(base, c)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    b_, c_ = cifras(base, n_cifras), cifras(c, n_cifras)
    print(f"{b_}^x = {c_}   →   x = ln({c_}) / ln({b_}) = {cifras(xs, n_cifras)}")
    print("comprobación: base^x =", cifras(base**xs, n_cifras))
    xx = np.linspace(xs - 3, xs + 3, 300)
    plt.figure(figsize=(4.5, 3)); plt.plot(xx, base**xx, "b-")
    plt.axhline(c, color="gray", ls="--"); plt.plot([xs], [c], "rs")
    plt.axhline(0, color="k", lw=0.8); plt.grid(True); plt.xlabel("x"); plt.ylabel("base^x"); plt.show()


widgets.interact(calculadora_exponencial,
    base=widgets.FloatText(value=5.0, description="base"), c=widgets.FloatText(value=7.0, description="c"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

interactive(children=(FloatText(value=5.0, description='base'), FloatText(value=7.0, description='c'), IntSlid…

In [25]:
def sinusoide(A, w, fi, k=0.0):
    """Parámetros de v(t) = A sen(w t + fi) + k, con w en rad/s y fi en rad."""
    if A == 0:
        raise ValueError("A = 0: la señal es constante y no tiene amplitud.")
    if w == 0:
        raise ValueError("ω = 0: la señal es constante y no tiene periodo.")
    return {"amplitud": abs(A), "periodo": 2*math.pi/abs(w), "frecuencia": abs(w)/(2*math.pi),
            "v(0)": A*math.sin(fi) + k, "máximo": k + abs(A), "mínimo": k - abs(A)}


def calculadora_sinusoide(A, w, fi, k, n_cifras):
    try:
        r = sinusoide(A, w, fi, k)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"amplitud = {cifras(r['amplitud'], n_cifras)}      periodo T = {cifras(r['periodo'], n_cifras)} s      "
          f"frecuencia f = {cifras(r['frecuencia'], n_cifras)} Hz")
    print(f"v(0) = {cifras(r['v(0)'], n_cifras)}      máximo = {cifras(r['máximo'], n_cifras)}      mínimo = {cifras(r['mínimo'], n_cifras)}")
    t = np.linspace(0, 2*r["periodo"], 600)
    plt.figure(figsize=(5, 3)); plt.plot(t, A*np.sin(w*t + fi) + k, "b-")
    plt.axhline(k, color="gray", ls="--"); plt.axhline(0, color="k", lw=0.8); plt.grid(True)
    plt.xlabel("t (s)"); plt.ylabel("v(t)"); plt.show()


widgets.interact(calculadora_sinusoide,
    A=widgets.FloatText(value=8.0, description="A"), w=widgets.FloatText(value=100*math.pi, description="ω (rad/s)"),
    fi=widgets.FloatText(value=math.pi/4, description="φ (rad)"), k=widgets.FloatText(value=0.0, description="k"),
    n_cifras=widgets.IntSlider(value=5, min=1, max=8, description="cifras sig."));

interactive(children=(FloatText(value=8.0, description='A'), FloatText(value=314.1592653589793, description='ω…

In [26]:
def cerca(u, v, tol=1e-9):
    return abs(u - v) < tol

PRUEBAS_6 = [
    ("sen 30° = 0.5 y cos 60° = 0.5",
     lambda: cerca(trig_en(30, "grados")[0], 0.5) and cerca(trig_en(60, "grados")[1], 0.5)),
    ("30 radianes no es 30 grados: sen(30 rad) = -0.988",
     lambda: cerca(trig_en(30, "radianes")[0], -0.9880316, 1e-6)),
    ("tan 90° no existe", lambda: trig_en(90, "grados")[2] is None),
    ("la unidad es obligatoria", lambda: _rechaza(lambda: trig_en(30, "gradianes"))),
    ("5^x = 7 da x = 1.2091", lambda: round(resolver_exponencial(5, 7), 4) == 1.2091),
    ("2^x = 40 da x = log2(40) = 5.3219", lambda: round(resolver_exponencial(2, 40), 4) == 5.3219),
    ("c ≤ 0 se rechaza con mensaje", lambda: _rechaza(lambda: resolver_exponencial(2, -3))),
    ("base 1 se rechaza con mensaje", lambda: _rechaza(lambda: resolver_exponencial(1, 5))),
    ("8 sen(100πt + π/4): T = 0.02 s, f = 50 Hz, v(0) = 5.657",
     lambda: (lambda r: cerca(r["periodo"], 0.02) and cerca(r["frecuencia"], 50.0)
              and round(r["v(0)"], 3) == 5.657)(sinusoide(8, 100*math.pi, math.pi/4))),
    ("dominio de ln(4 - x²) es (-2, 2)",
     lambda: continuous_domain(parsear("log(4 - x**2)"), x, sp.S.Reals) == sp.Interval.open(-2, 2)),
    ("log2(40) - log2(5) = 3", lambda: cerca(math.log2(40) - math.log2(5), 3.0)),
]

for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

OK   sen 30° = 0.5 y cos 60° = 0.5
OK   30 radianes no es 30 grados: sen(30 rad) = -0.988
OK   tan 90° no existe
OK   la unidad es obligatoria
OK   5^x = 7 da x = 1.2091
OK   2^x = 40 da x = log2(40) = 5.3219
OK   c ≤ 0 se rechaza con mensaje
OK   base 1 se rechaza con mensaje
OK   8 sen(100πt + π/4): T = 0.02 s, f = 50 Hz, v(0) = 5.657
OK   dominio de ln(4 - x²) es (-2, 2)
OK   log2(40) - log2(5) = 3


**Contrasta (sección 6).** Calcula a mano $\log_2 40-\log_2 5$ y $\mathrm{sen}\,150^\circ$ (usa el ángulo de referencia). Escribe tus resultados y compáralos con la calculadora. Anota: ¿usaste la propiedad del cociente? ¿de qué cuadrante salió el signo?

In [27]:
mi_log = None          # por ejemplo: 3
mi_seno = None         # por ejemplo: 0.5

esperado_log = math.log2(40) - math.log2(5)
esperado_seno = trig_en(150, "grados")[0]
print("La calculadora da: log2(40) - log2(5) =", round(esperado_log, 6), "| sen 150° =", round(esperado_seno, 6))
for nombre, mio, ref in (("log2(40) - log2(5)", mi_log, esperado_log), ("sen 150°", mi_seno, esperado_seno)):
    if mio is None:
        print(f"{nombre}: falta tu cálculo a mano.")
    else:
        print(f"{nombre}:", "coincide" if cerca(mio, ref, 1e-6) else "NO coincide, revisa tu paso")

La calculadora da: log2(40) - log2(5) = 3.0 | sen 150° = 0.5
log2(40) - log2(5): falta tu cálculo a mano.
sen 150°: falta tu cálculo a mano.


## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [28]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5 + PRUEBAS_6
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")

Todas las verificaciones pasaron: 39/39
